# TSP demo: CoolTsp with pickup/delivery

This notebook loads a Solomon-format instance (e.g. `uniform_300.txt` or `c101.txt`) as a **CoolTsp** problem and solves TSP with added "pickup" twist. The CoolSolver accepts various TSP solvers as dependency injection, as shown here (Naive, Nearest-Neighbor, 2-opt).

**80/20 split:** When we call `load_cool_bench(path, seed=...)`, it reads all customer nodes (coordinates and demand) and vehicle capacity, then **randomly assigns 20% of nodes as pickups and 80% as deliveries** (using the given seed for reproducibility). The solver first plans a delivery tour that respects capacity (affected by DI TSP solvers); it will then insert one pickup along the route. See [algo_explainer.md](../algo_explainer.md) for the pickup-detour algorithm.

In [ ]:
%matplotlib inline
import sys

sys.path.insert(0, "bench")

from mytsp import (
    CoolSolver,
    NaiveSolver,
    NearestNeighborSolver,
    Simple2PointSolver,
    load_cool_bench,
)
from viz import plot_cool_route

**Load an instance:** The next cell loads a Solomon-format file (e.g. `bench/uniform_300.txt`). `load_cool_bench` parses coordinates and demand, then applies the **80/20 split** (20% pickups, 80% deliveries) with a fixed seed so results are reproducible. We then solve with `CoolSolver(NaiveSolver())` as a baseline.

In [ ]:
from pathlib import Path

p = Path("bench/uniform_300.txt")
input_path = p if p.exists() else Path("uniform_300.txt")
problem = load_cool_bench(input_path)
problem.vehicle_capacity = 100
solver = CoolSolver(NaiveSolver())
result = solver.solve(problem)
d, p, dist = len(result.nodes), len(result.unused_nodes), result.solution.distance
print(f"Deliveries: {d}, Pickups (unused): {p}, Distance: {dist:.2f}")

In [ ]:
plot_cool_route(
    result,
    title=f"CoolSolver(NaiveSolver()) — tour length: {result.solution.distance:.2f}",
)

In [ ]:
nn_solver = CoolSolver(NearestNeighborSolver())
nn_result = nn_solver.solve(problem)
gd = len(nn_result.nodes)
gp = len(nn_result.unused_nodes)
gdist = nn_result.solution.distance
print(f"NN - Deliveries: {gd}, Pickups (unused): {gp}, Distance: {gdist:.2f}")

In [ ]:
plot_cool_route(
    nn_result,
    title=(
        f"CoolSolver(NearestNeighborSolver()) — tour length: "
        f"{nn_result.solution.distance:.2f}"
    ),
)

In [ ]:
two_opt_solver = CoolSolver(Simple2PointSolver())
two_opt_result = two_opt_solver.solve(problem)
inner = two_opt_solver._tsp_solver
td = len(two_opt_result.nodes)
tp = len(two_opt_result.unused_nodes)
tdist = two_opt_result.solution.distance
print(f"2-opt - Deliveries: {td}, Pickups (unused): {tp}, Distance: {tdist:.2f}")
print("Solver stats:")
for k, v in inner.stats.items():
    if k != "distance_history":
        print(f"  {k}: {v}")
    else:
        tail = v[-1] if v else "N/A"
        print(f"  {k}: length {len(v)} (first 5: {v[:5]} ... last: {tail})")

In [ ]:
plot_cool_route(
    two_opt_result,
    title=(
        f"CoolSolver(Simple2PointSolver()) — tour length: "
        f"{two_opt_result.solution.distance:.2f}"
    ),
)